## Imports

In [ ]:
import sys
sys.path.append("..")

from dotenv import load_dotenv
load_dotenv("../.env")

from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas import evaluate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from datasets import Dataset
import pandas as pd

from app.core.retrieval import retrieve
from app.core.generation import generate, create_chat_history
from app.config import settings

## Define test questions and references

In [ ]:
testset_df = pd.DataFrame([
    {"user_input": "What is the full year FY23 revenue guidance?",
     "reference": "$31.7 - $31.8 Billion"},
    {"user_input": "What is the Q2 FY23 revenue guidance?",
     "reference": "$7.69 - $7.70 Billion"},
    {"user_input": "What is the FX impact on full year revenue?",
     "reference": "~($600M) year over year FX impact"},
    {"user_input": "What is the non-GAAP operating margin guidance?",
     "reference": "~20.4%"},
    {"user_input": "What is the GAAP operating margin guidance?",
     "reference": "~3.8%"},
    {"user_input": "What is the operating cash flow growth guidance?",
     "reference": "~21% - 22%"},
    {"user_input": "What is the current remaining performance obligation?",
     "reference": "Approximately $21.5 billion, an increase of 21% year-over-year"},
    {"user_input": "What assumptions does the guidance make about the investment portfolio?",
     "reference": "Guidance assumes no change to the value of the strategic investment portfolio"},
    {"user_input": "When is the earnings call scheduled?",
     "reference": "May 31, 2022 at 2:00 PM Pacific Time"},
    {"user_input": "What is the GAAP earnings per share guidance for Q2?",
     "reference": "($0.03) - ($0.02)"},
])

print(testset_df)

## RAG pipeline runner

In [ ]:
def run_pipeline(questions, references, namespace, rerank=True):
    results = []
    chat_history = create_chat_history()

    for question, reference in zip(questions, references):
        chunks = retrieve(question, namespace=namespace, rerank=rerank)
        answer = generate(question, chunks, chat_history)
        results.append({
            "user_input": question,
            "response": answer,
            "retrieved_contexts": [c["text"] for c in chunks],
            "reference": reference
        })
    return results

## Run eval — dense only vs dense + rerank

In [ ]:
namespace = "salesforcefinancial.pdf"
questions = testset_df["user_input"].tolist()
references = testset_df["reference"].tolist()

print("Running dense only...")
dense_results = run_pipeline(questions, references, namespace, rerank=False)

print("Running dense + rerank...")
rerank_results = run_pipeline(questions, references, namespace, rerank=True)

print("Done — running RAGAS eval...")

eval_llm = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-4o-mini", api_key=settings.openai_api_key)
)
eval_embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(model=settings.embedding_model, api_key=settings.openai_api_key)
)

metrics = [faithfulness, answer_relevancy, context_precision, context_recall]

dense_dataset = Dataset.from_list(dense_results)
rerank_dataset = Dataset.from_list(rerank_results)

dense_scores = evaluate(dense_dataset, metrics=metrics, llm=eval_llm, embeddings=eval_embeddings)
rerank_scores = evaluate(rerank_dataset, metrics=metrics, llm=eval_llm, embeddings=eval_embeddings)

print("Dense only:", dense_scores)
print("Dense + Rerank:", rerank_scores)

## Results table

In [ ]:
results_df = pd.DataFrame({
    "Configuration": ["Dense only", "Dense + Cohere rerank"],
    "Faithfulness": [
        dense_scores["faithfulness"],
        rerank_scores["faithfulness"]
    ],
    "Context Precision": [
        dense_scores["context_precision"],
        rerank_scores["context_precision"]
    ],
    "Context Recall": [
        dense_scores["context_recall"],
        rerank_scores["context_recall"]
    ],
    "Answer Relevancy": [
        dense_scores["answer_relevancy"],
        rerank_scores["answer_relevancy"]
    ]
})

print(results_df.to_string(index=False))
results_df.to_csv("eval_results.csv", index=False)